In [2]:
import requests
# import pandas as pd

from datetime import datetime, timedelta


In [3]:

import sys
import os

root = os.path.abspath(os.path.join(os.getcwd(), "../"))

if root not in sys.path:
    sys.path.insert(0, root)

print(root)

/home/hilaneto/Jhan/Trabalho/python/projetos/AtualizaIndicador


In [ ]:

# Datas -------------------------------------------------------
hoje = datetime.now()

data_inicial = (hoje - timedelta(days=230)).strftime("%d/%m/%Y")
data_final = hoje.strftime("%d/%m/%Y")

data_inicial = '01/01/2026'
data_final = '28/01/2026'

print(f"Período: {data_inicial} até {data_final}")

In [ ]:

# https://api.bcb.gov.br/dados/serie/bcdata.sgs.10844/dados?formato=json&dataInicial=25/06/2026&dataFinal=18/08/2026

# API Banco Central -------------------------------------------
url = "https://api.bcb.gov.br/dados/serie/bcdata.sgs.432/dados"

parametros = { "formato": "json", "dataInicial": data_inicial, "dataFinal": data_final }

resposta = requests.get(url, params=parametros, timeout=10)
resposta.raise_for_status()
dados = resposta.json() # lista de dicionários

# Percorrer a lista de dicionário
for selic in dados:
    print(selic)


In [ ]:
for selic in dados:
    print(selic["data"], selic["valor"])


In [ ]:

from database.conexao import conectar
from indicadores.selic import Selic
from decimal import Decimal

dados_selic = []

with conectar():
    
    for registro in dados:
        selic = {"indice": Decimal(registro["valor"]),
                "status": True,
                "dt_referencia": datetime.strptime(registro["data"],"%d/%m/%Y").date(),
                "dt_atualizacao": datetime.now()}
        dados_selic.append(selic)
        
        Selic.insert(**selic).on_conflict(
        conflict_target=[Selic.dt_referencia],
        update={Selic.indice: selic["indice"],
                Selic.status: selic["status"],
                Selic.dt_atualizacao: selic["dt_atualizacao"]}).execute()


In [ ]:

from database.conexao import conectar
from indicadores.selic import Selic

with conectar():
    dados_selic = Selic.buscar()
    print(dados_selic)


In [8]:

from database.conexao import conectar
from indicadores.selic import Selic
from datetime import datetime
from decimal import Decimal

dados_teste = [
    {"dt_referencia": "18/03/2026", "indice": "15.00"},
    {"dt_referencia": "29/04/2026", "indice": "14.75"},
    {"dt_referencia": "17/06/2026", "indice": "14.50"},
    {"dt_referencia": "05/08/2026", "indice": "14.25"},
    {"dt_referencia": "28/08/2026", "indice": "14.00"},
]

with conectar():
    for registro in dados_teste:
        Selic.create(
            indice=Decimal(registro["indice"]),
            status=True,
            dt_referencia=datetime.strptime(
                registro["dt_referencia"],
                "%d/%m/%Y"
            ).date(),
            dt_atualizacao=datetime.now()
        )

In [7]:

from database.conexao import conectar
from indicadores.selic import Selic

aa = Selic.atualizar_selic()

print(aa)

None
